# 第5章形式 MiniMax Music 3をColabで動かす

公開重みの **MiniMaxAI/MiniMax-Music3** をGoogle Colabに読み込み、
通常の関数呼び出しによる楽曲生成と、Gradioを用いたブラウザUIを実行します。

このNotebookは、

1. 実行環境の準備
2. Google DriveとHugging Faceキャッシュ設定
3. MiniMax Music 3の読み込みと生成関数
4. 短い楽曲で動作確認
5. Gradio UI

の順に実行します。

> **推奨環境**
>
> - 第一候補: Google Colab有料版 + L4 24GB / A100
> - 実験候補: 無料版T4 16GB（Low VRAMモードを使用）
>
> MiniMax公式は、通常構成を24GB以上、automatic CPU offloadで約22GB、
> language modelをlayer-by-layerでoffloadする構成では8GB GPUまで収まると説明しています。
> ただしT4ではCPU/GPU間転送が多くなり、生成はかなり遅くなる可能性があります。
>
> **実行方針**
>
> GradioとTorchはColab既定版を基本的に利用します。
> MiniMax Music 3対応がDiffusers本流へ完全反映されるまで、
> 公式モデルカードで案内されているDiffusersのコミットをインストールします。

In [1]:
# =========================================
# コード5-M3-1 実行環境の準備
# =========================================
!nvidia-smi -L || echo "No GPU"
!python -V

# 2026-08-15時点のMiniMax公式モデルカードが案内するDiffusers実装。
%pip -q install -U git+https://github.com/huggingface/diffusers@dafe3733fcfdbf3c48915fe77be3aef65b5d6a2d transformers accelerate soundfile

import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPUランタイムを有効にしてください。")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

print("GPU:", gpu_name)
print(f"VRAM: {gpu_mem_gb:.1f} GB")
print("BF16 supported:", torch.cuda.is_bf16_supported())

GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition (UUID: GPU-89c32cda-6871-dc3e-2596-176e4e81fcd1)
Python 3.12.13
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 174.0 MB/s eta 0:00:00
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 95.0 GB
BF16 supported: True


In [2]:
# =========================================
from google.colab import drive
from pathlib import Path
import os
import shutil

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/LocalLLM")
USE_DRIVE_CACHE = True

if USE_DRIVE_CACHE:
    CACHE_DIR = PROJECT_DIR / "hf_cache"
    OUTPUT_DIR = PROJECT_DIR / "outputs"
else:
    CACHE_DIR = Path("/content/hf_cache")
    OUTPUT_DIR = Path("/content/minimax_music3_outputs")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(CACHE_DIR)

os.environ["HF_HUB_CACHE"] = str(CACHE_DIR)

usage = shutil.disk_usage(CACHE_DIR)
print("CACHE_DIR:", CACHE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print(f"free space: {usage.free / 1024**3:.1f} GB")

if usage.free < 35 * 1024**3:
    print("WARNING: MiniMax Music 3は大規模です。十分な空き容量を確保してください。")

Mounted at /content/drive
CACHE_DIR: /content/drive/MyDrive/Colab Notebooks/LocalLLM/hf_cache
OUTPUT_DIR: /content/drive/MyDrive/Colab Notebooks/LocalLLM/outputs
free space: 179.0 GB


In [6]:
# =========================================
# コード5-M3-4 MiniMax Music 3モデルと楽曲生成関数
# =========================================
import gc
import time
from datetime import datetime

import numpy as np
import soundfile as sf
import torch
from diffusers import ComponentsManager, ModularPipeline
from diffusers.hooks import apply_group_offloading

MODEL_ID = "MiniMaxAI/MiniMax-Music3"

# "auto" : VRAM 22GB未満ならLow VRAM
# True   : 常にLow VRAM
# False  : 通常ロード
LOW_VRAM_MODE = "auto"

gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

if LOW_VRAM_MODE == "auto":
    use_low_vram = gpu_mem_gb < 22.0
else:
    use_low_vram = bool(LOW_VRAM_MODE)

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print("MODEL_ID:", MODEL_ID)
print("dtype:", DTYPE)
print("Low VRAM mode:", use_low_vram)

if use_low_vram:
    manager = ComponentsManager()
    manager.enable_auto_cpu_offload(device="cuda")

    pipe = ModularPipeline.from_pretrained(
        MODEL_ID,
        components_manager=manager,
        cache_dir=str(CACHE_DIR),
    )
    pipe.load_components(dtype=DTYPE)

    apply_group_offloading(
        pipe.language_model,
        onload_device=torch.device("cuda"),
        offload_type="leaf_level",
        use_stream=True,
    )

else:
    pipe = ModularPipeline.from_pretrained(
        MODEL_ID,
        cache_dir=str(CACHE_DIR),
    )
    pipe.load_components(dtype=DTYPE)
    pipe.to("cuda")

print("sampling rate:", pipe.sampling_rate)


@torch.inference_mode()
def music_generate(
    prompt,
    lyrics,
    audio_duration=30.0,
    seed=7,
):
    prompt = (prompt or "").strip()
    lyrics = (lyrics or "").strip()

    if not prompt:
        raise ValueError("Music Descriptionを入力してください。")

    if not lyrics:
        raise ValueError("Lyricsを入力してください。")

    audio_duration = float(audio_duration)
    seed = int(seed)

    if audio_duration < 5:
        raise ValueError("audio_durationは5秒以上にしてください。")

    if audio_duration > 300:
        raise ValueError("このNotebookでは最大300秒に制限しています。")

    gc.collect()
    torch.cuda.empty_cache()

    start = time.time()

    audio = pipe(
        prompt=prompt,
        lyrics=lyrics,
        audio_duration=audio_duration,
        generator=torch.Generator("cuda").manual_seed(seed),
        output="audios",
    )[0]

    elapsed = time.time() - start

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_file = OUTPUT_DIR / f"minimax_music3_{stamp}_seed{seed}.wav"

    # MiniMax Music 3はnumpy.ndarrayを返す場合がある
    if torch.is_tensor(audio):
        audio_np = audio.detach().float().cpu().numpy()
    else:
        audio_np = np.asarray(audio)

    sf.write(
        str(out_file),
        audio_np.T,
        pipe.sampling_rate,
    )

    print(f"saved: {out_file}")
    print(f"generation time: {elapsed:.1f} sec")

    return str(out_file), elapsed

Guiders are currently an experimental feature under active development. The API is subject to breaking changes in future releases.


MODEL_ID: MiniMaxAI/MiniMax-Music3
dtype: torch.bfloat16
Low VRAM mode: False


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

sampling rate: 44100


In [13]:
# =========================================
# コード5-M3-4 動作確認
# =========================================
# 最初は短めの生成で確認してください。
# T4 Low VRAMでは特に時間がかかる可能性があります。

test_lyrics = """[Verse]
夜明けの街を歩いて
まだ見えない明日を探す

[Chorus]
光の向こうへ
もう一度走り出そう
"""

test_prompt = (
    "Genre: Japanese electronic pop. BPM: 128. Key: A minor. "
    "Emotional and energetic female lead vocal with clear articulation. "
    "Arrangement: bright synthesizers, punchy electronic drums, warm bass, "
    "and a wide uplifting chorus with layered backing vocals."
)

test_file, test_elapsed = music_generate(
    prompt=test_prompt,
    lyrics=test_lyrics,
    audio_duration=60.0,
    seed=7,
)

print("OUTPUT:", test_file)
print(f"elapsed: {test_elapsed:.1f} sec")

from IPython.display import Audio, display
display(Audio(test_file))

Output hidden; open in https://colab.research.google.com to view.

In [14]:
# =========================================
# コード5-M3-5 Gradioを用いたMiniMax Music 3 UI
# =========================================
import gradio as gr

print("gradio:", gr.__version__)

DEFAULT_PROMPT = """Genre: Japanese electronic pop.
BPM: 128.
Key: A minor.
Mood: emotional, futuristic, energetic.
Vocals: clear female lead vocal, expressive but controlled, layered harmonies in the chorus.
Arrangement: bright synthesizers, arpeggiated synths, punchy electronic drums, warm bass,
a restrained verse, rising pre-chorus, and a wide powerful chorus.
Production: modern polished stereo mix with clear vocals and spacious effects."""

DEFAULT_LYRICS = """[Verse]
夜明けの街を歩いて
まだ見えない明日を探す

[Pre-Chorus]
小さな光を集めて
もう一度息を吸う

[Chorus]
光の向こうへ
迷いながら走り出そう
新しい景色が
この先で待っている
"""

def gr_music_generate(prompt, lyrics, duration, seed):
    try:
        out_file, elapsed = music_generate(
            prompt=prompt,
            lyrics=lyrics,
            audio_duration=float(duration),
            seed=int(seed),
        )
        status = (
            f"生成完了: {elapsed:.1f}秒 / "
            f"GPU: {torch.cuda.get_device_name(0)} / "
            f"Low VRAM: {use_low_vram}"
        )
        return out_file, status
    except Exception as e:
        return None, f"ERROR: {type(e).__name__}: {e}"

with gr.Blocks(title="MiniMax Music 3") as music_demo:
    gr.Markdown(
        "## MiniMax Music 3 - Google Colab\n"
        "公開重みをColab上で実行し、歌詞とMusic Descriptionから楽曲を生成します。"
    )

    with gr.Row():
        with gr.Column():
            prompt_box = gr.Textbox(
                value=DEFAULT_PROMPT,
                label="Music Description",
                lines=12,
            )
            lyrics_box = gr.Textbox(
                value=DEFAULT_LYRICS,
                label="Lyrics",
                lines=14,
            )

            with gr.Row():
                duration_slider = gr.Slider(
                    minimum=10,
                    maximum=120,
                    value=30,
                    step=5,
                    label="Duration (seconds)",
                )
                seed_number = gr.Number(
                    value=7,
                    precision=0,
                    label="Seed",
                )

            generate_btn = gr.Button("Generate Music", variant="primary")

        with gr.Column():
            audio_output = gr.Audio(
                label="Generated Music",
                type="filepath",
            )
            status_box = gr.Textbox(
                label="Status",
                interactive=False,
            )

    generate_btn.click(
        gr_music_generate,
        inputs=[prompt_box, lyrics_box, duration_slider, seed_number],
        outputs=[audio_output, status_box],
    )

music_demo.queue(default_concurrency_limit=1)
music_demo.launch(share=True, debug=False)

gradio: 6.20.0
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2322bb71ad1af2ccb6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 実行時の目安と注意

### 有料版Colab

L4 24GB以上を第一候補とします。VRAMが22GB以上なら通常ロード、
それ未満ならNotebookが自動的にLow VRAMモードを選びます。

### 無料版Colab / T4 16GB

MiniMax公式のLow VRAM構成では8GB GPUまで収まるとされています。
そのためT4 16GBでも**メモリ容量だけを見れば実行候補**です。

ただし、T4では以下の点に注意してください。

- language modelの層をCPUとGPUの間で入れ替えるため遅い
- Colab側のCPU RAMやストレージ条件にも影響される
- 長い楽曲ほど生成時間が大きくなる
- 最初は10〜20秒程度で動作確認する
- 無料版ColabのGPU割り当てや利用上限は保証されない

したがって本Notebookでは、

**L4/A100で動作確認 → T4 Low VRAMで限界を試す**

という順序を推奨します。

### プロンプト

MiniMax Music 3は、

- Lyrics
- Music Description

の2入力を受け取ります。

Lyricsには `[Verse]`, `[Chorus]`, `[Bridge]`, `[Instrumental]`, `[Outro]`
などのセクションタグを利用できます。

Music DescriptionではGenre、BPM、Key、Mood、Vocals、Arrangement、
Productionなどを具体的に書くほど細かく制御できます。

### 出力

32 kHz / 16-bit stereo WAVとしてGoogle Driveまたは`/content`へ保存します。